# SymboLM v2.0 — Random 200-Question Multi-Year Benchmark Evaluation on GPU
**Author & Principal Architect:** Sanish Gyawali (Rupandehi, Nepal)  
**AI Systems Collaborator / Operational Exoskeleton:** Antigravity (Google DeepMind)  
**Repository:** [https://github.com/gyawalisanish0/symboLM](https://github.com/gyawalisanish0/symboLM)  

### Objective:
Evaluates the freshly trained **SymboLM v2.0 Contrastive SFT policy adapter** across **200 stratified random questions** sampled from the 2021–2026 multi-year benchmark suite (GSM8K, AMC 10/12, MATH, AIME, and Modern Math 2026).  
Measures:
1. **Pass@1 Accuracy**
2. **Token Generation & Compression vs. English CoT Baseline**
3. **Register Routing Fidelity** (`<reg:bypass>` vs `<reg:plan>`/`<reg:deduce>` vs `<reg:intent>`)
4. **Information Efficiency Ratio (IER)**
5. **Trailing Syntax Cleanliness**

In [ ]:
# Cell 1: Environment & GPU Hardware Verification
import os
import sys
import torch

print('Python version:', sys.version)
print('Torch version :', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB VRAM)')
else:
    print('[Warning] No CUDA GPU found!')


In [ ]:
# Cell 2: Dependencies Installation & Environment Setup
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q 'transformers>=4.48.0' 'peft>=0.14.0' 'accelerate>=1.2.0' 'sympy>=1.12' 'datasets>=3.2.0'


In [ ]:
# Cell 3: Clone latest SymboLM repository from GitHub
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM
!git log -n 1 --oneline


In [ ]:
# Cell 4: Mount SymboLM v2.0 Policy Adapter from Kernel Source
import os
import shutil
from pathlib import Path

target_dir = Path('/kaggle/working/symboLM/checkpoints/symbolm_v2_adapter')
target_dir.mkdir(parents=True, exist_ok=True)

found = False
# 1. Search for uncompressed adapter directory in mounted kernel sources
for p in Path('/kaggle/input').glob('**/symbolm_v2_adapter'):
    if (p / 'adapter_config.json').exists():
        print(f'[OK] Found uncompressed v2 adapter at: {p}')
        for item in p.iterdir():
            if item.is_file():
                shutil.copy2(item, target_dir / item.name)
        found = True
        break

# 2. Search for v2 policy tarball archive
if not found:
    for arc in Path('/kaggle/input').glob('**/*v2*contrastive*policy*.tar.gz'):
        print(f'[OK] Found v2 policy tar archive at: {arc}. Unpacking...')
        os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
        found = True
        break

print('[Status] Validating SymboLM v2 Adapter Files in target:')
for f in sorted(target_dir.iterdir()):
    sz = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name} ({sz:.2f} MB)' if f.is_file() else f'  {f.name}/')

assert (target_dir / 'adapter_config.json').exists(), 'CRITICAL: adapter_config.json missing!'
assert (target_dir / 'adapter_model.safetensors').exists(), 'CRITICAL: adapter_model.safetensors missing!'
print('[VERIFIED] SymboLM v2.0 Policy Adapter is ready for 200-question evaluation!')


In [ ]:
# Cell 5: Generate Stratified Random 200-Question Multi-Year Benchmark
!python -m data.sample_random_200
!sha256sum data/eval_200_random_multi_year.jsonl


In [ ]:
# Cell 6: Run 200-Question Evaluation on NVIDIA GPU
!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/symboLM/eval/run_200_mixed_benchmark.py \
  --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
  --adapter_path /kaggle/working/symboLM/checkpoints/symbolm_v2_adapter \
  --data_path /kaggle/working/symboLM/data/eval_200_random_multi_year.jsonl \
  --output_path /kaggle/working/eval_v2_200_results.json \
  --report_path /kaggle/working/EVAL_V2_200_REPORT.md \
  --max_tokens 192 \
  --temperature 0.2


In [ ]:
# Cell 7: Display Full Scorecard and Audit Receipts
import os
rep_path = '/kaggle/working/EVAL_V2_200_REPORT.md'
if os.path.exists(rep_path):
    with open(rep_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print('[Warning] Report file not found!')


In [ ]:
# Cell 8: Package Evaluation Receipts and Scorecards
!tar -czf /kaggle/working/symbolm_v2_200_eval_receipts.tar.gz -C /kaggle/working eval_v2_200_results.json EVAL_V2_200_REPORT.md
print('[COMPLETE] Packaged receipts into /kaggle/working/symbolm_v2_200_eval_receipts.tar.gz')
